In [ ]:
# Шаг 0 — прочитайте и запустите эту ячейку (покажет инструкцию).
from IPython.display import Markdown, display

display(
    Markdown(
        """
# MinerU в Google Colab

**Сделайте:** *Runtime → Change runtime type → GPU*, затем выполняйте **следующие ячейки по порядку** (каждая — код, можно Shift+Enter).

| # | Что делает ячейка |
|---|-------------------|
| 1 | Клон репозитория + `jiwer` |
| 2 | Установка `mineru` (долго) |
| 3 | Проверка путей, PNG, эталонов |
| 4 | Опционально ModelScope |
| 5 | Запуск `mineru_image_benchmark.py` |

**Артефакты:** `output/mineru_benchmark/…` — см. вывод последней ячейки.

Форк: переменные `OCR_ANALYZE_GIT_URL`, `OCR_ANALYZE_COLAB_DIR`. После первой установки MinerU часто нужен **Restart runtime**, затем снова ячейки 1–3 и 5.
"""
    )
)
print("OK: дальше ячейка 1 (клон + jiwer).")


In [ ]:
# Шаг 1 — клон репозитория + jiwer (нужен скрипту метрик).
# Colab: клонирует в REPO_DIR и делает chdir. Локально: только pip jiwer из текущего каталога.

from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


GIT_URL = os.environ.get(
    "OCR_ANALYZE_GIT_URL",
    "https://github.com/developer-mixa/OCR-Analyze.git",
)
REPO_DIR = Path(os.environ.get("OCR_ANALYZE_COLAB_DIR", "/content/OCR-Analyze"))

if in_colab():
    if not (REPO_DIR / "scripts").is_dir():
        print("Клонирую", GIT_URL, "→", REPO_DIR)
        subprocess.check_call(["git", "clone", "--depth", "1", GIT_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)
    print("Рабочий каталог:", Path.cwd().resolve())
else:
    print("Не Colab — клон не выполняется. cwd:", Path.cwd().resolve())

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "jiwer"])
print("OK: jiwer. Следующая ячейка — установка MinerU (долго).")


In [ ]:
# Шаг 2 — установка MinerU (долго, много места на диске).
# По умолчанию mineru[pipeline] (легче). Полный набор: в ячейке ВЫШЕ выполните
#   import os; os.environ["MINERU_PKG"] = "all"
# Затем снова запустите эту ячейку.

import os
import shutil
import subprocess
import sys
from pathlib import Path

PKG = os.environ.get("MINERU_PKG", "pipeline").strip().lower()
if PKG not in ("pipeline", "all"):
    raise ValueError("MINERU_PKG должен быть pipeline или all")

spec = "mineru[pipeline]" if PKG == "pipeline" else "mineru[all]"
print("Устанавливаю", spec, "...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", spec])
print("which(mineru) =", shutil.which("mineru"))
print("Если None → Runtime → Restart session, затем снова ячейки 1–2.")


In [ ]:
# Шаг 3 — корень репозитория, PNG и эталоны рядом с ними.

from __future__ import annotations

from pathlib import Path

REPO_ROOT_OVERRIDE: Path | None = None


def find_repo_root() -> Path:
    if REPO_ROOT_OVERRIDE is not None:
        p = REPO_ROOT_OVERRIDE.expanduser().resolve()
        if (p / "scripts").is_dir():
            return p
    cwd = Path.cwd().resolve()
    for start in [cwd, *cwd.parents]:
        if (start / "scripts" / "mineru_image_benchmark.py").is_file():
            return start
    return cwd


REPO_ROOT = find_repo_root()
INPUT_DIR = REPO_ROOT / "input" / "data" / "1"
SCRIPT = REPO_ROOT / "scripts" / "mineru_image_benchmark.py"
OUT_DIR = REPO_ROOT / "output" / "mineru_benchmark"

print("REPO_ROOT =", REPO_ROOT.resolve())
print("SCRIPT:", SCRIPT.is_file(), SCRIPT)
print("INPUT_DIR:", INPUT_DIR.is_dir(), INPUT_DIR)

_png = sorted(INPUT_DIR.glob("*.png")) if INPUT_DIR.is_dir() else []
print("PNG:", len(_png))
for p in _png:
    stem = p.stem
    refs = [n for n in (f"{stem}.ref.txt", f"{stem}.ref.md", f"{stem}.txt", f"{stem}.md") if (INPUT_DIR / n).is_file()]
    print(" ", p.name, "| эталон:", ", ".join(refs) if refs else "нет")
if not _png:
    print("Нет PNG — добавьте в input/data/1")


In [ ]:
# Шаг 4 (опционально) — ModelScope вместо Hugging Face для загрузки весов MinerU.
# Чтобы включить: в ячейке ВЫШЕ выполните  import os; os.environ["USE_MODELSCOPE_FOR_MINERU"]="1"
# Затем запустите эту ячейку. Док: https://opendatalab.github.io/MinerU/usage/quick_usage/

import os

if os.environ.get("USE_MODELSCOPE_FOR_MINERU", "").strip() == "1":
    os.environ["MINERU_MODEL_SOURCE"] = "modelscope"
    print("Включено MINERU_MODEL_SOURCE =", os.environ["MINERU_MODEL_SOURCE"])
else:
    print("Пропуск (нормально). Для ModelScope задайте USE_MODELSCOPE_FOR_MINERU=1 в ячейке выше.")


In [ ]:
# Шаг 5 — прогон mineru_image_benchmark.py (mineru → .md → CER к эталонам).
# Параметры: переменные MINERU_BACKEND, MINERU_LANG, MINERU_METHOD или правка списка cmd ниже.

import os
import shutil
import subprocess
import sys
from pathlib import Path

if "REPO_ROOT" not in globals() or "SCRIPT" not in globals():
    raise RuntimeError("Сначала выполните шаг 3 (ячейка с REPO_ROOT).")

if not shutil.which("mineru"):
    raise RuntimeError("mineru не в PATH — выполните шаг 2 и при необходимости Restart runtime.")

cmd = [
    sys.executable,
    str(SCRIPT),
    "--input-dir",
    str(INPUT_DIR),
    "--output-dir",
    str(OUT_DIR),
    "--backend",
    os.environ.get("MINERU_BACKEND", "pipeline"),
    "--lang",
    os.environ.get("MINERU_LANG", "cyrillic"),
]
if os.environ.get("MINERU_METHOD"):
    cmd.extend(["--method", os.environ["MINERU_METHOD"]])

print("Запуск:", " ".join(cmd))
subprocess.check_call(cmd, cwd=str(REPO_ROOT))

summ = OUT_DIR / "mineru_summaries.json"
if summ.is_file():
    print("\n---", summ.name, "---\n")
    print(summ.read_text(encoding="utf-8"))
